# 02 · Data preprocessing

**Guideline covered:** Data preprocessing

**What this notebook does.** Cleans the review text while protecting its meaning. Three decisions matter most:

| Decision | Why it matters |
|---|---|
| **Keep negations** ("not", "no", "never") | Standard stop-word lists delete "not", turning "payment *not* received" into "payment received". |
| **Translate Hinglish** with a dictionary of about 150 words, slang terms and brand spellings | 10–14% of long reviews mix Hindi and English ("paisa kat gaya" = "money deducted"); deleting them would lose many serious complaints. |
| **Two text versions** | `text_light` (lightly cleaned, natural sentences) for transformer models; `text_full` (lemmatised, stop words and emojis removed, negations kept) for word-counting methods. |

Other steps: contractions expanded, emojis stored in a separate column, junk reviews removed, reviews written only in Indian scripts flagged (kept for rating statistics, excluded from English text models).

**Cleaning funnel:** 86,056 scraped → 85,979 kept (77 junk removed) → 85,150 English-text reviews for modelling → 21,551 long reviews (5+ words).

**Inputs:** `data/raw/raw_reviews_v2.csv` · **Outputs:** `data/processed/clean_reviews.parquet`

**How to run:** open in Google Colab, keep the project folder at `MyDrive/upi-complaint-analytics` (or change `BASE` in the first code cell) and run the cells from top to bottom. The outputs shown are from the original run.

Cell P1: load the data and set up the dictionaries

In [ ]:
!pip install -q emoji
import re, emoji, pandas as pd, numpy as np, nltk, spacy
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

BASE = '/content/drive/MyDrive/upi-complaint-analytics'
df = pd.read_csv(f'{BASE}/data/raw/raw_reviews_v2.csv', parse_dates=['review_date', 'reply_date'])
df['review_text'] = df['review_text'].fillna('').astype(str)

CONTRACTIONS = [(r"won't", 'will not'), (r"can't", 'can not'), (r"n't\b", ' not'),
                (r"'re\b", ' are'), (r"'ll\b", ' will'), (r"'ve\b", ' have'),
                (r"'m\b", ' am'), (r"'d\b", ' would'), (r"'s\b", '')]

SLANG = {
    'dont': 'do not', 'cant': 'can not', 'wont': 'will not', 'doesnt': 'does not', 'didnt': 'did not',
    'isnt': 'is not', 'wasnt': 'was not', 'arent': 'are not', 'havent': 'have not', 'hasnt': 'has not',
    'couldnt': 'could not', 'shouldnt': 'should not', 'wouldnt': 'would not', 'nt': 'not',
    'pls': 'please', 'plz': 'please', 'plzz': 'please', 'plss': 'please',
    'thx': 'thanks', 'thnx': 'thanks', 'thanku': 'thanks', 'tq': 'thanks',
    'u': 'you', 'ur': 'your', 'msg': 'message', 'acc': 'account', 'acct': 'account', 'amt': 'amount',
    'txn': 'transaction', 'txns': 'transactions', 'trxn': 'transaction', 'transection': 'transaction',
    'transcation': 'transaction', 'tranaction': 'transaction', 'trasaction': 'transaction',
    'transction': 'transaction', 'paymnt': 'payment', 'pyment': 'payment', 'paymet': 'payment',
    'recived': 'received', 'recieved': 'received', 'receved': 'received',
    'wrking': 'working', 'workng': 'working', 'gud': 'good', 'vry': 'very', 'bt': 'but',
}
HINGLISH = {
    'nahi aaya': 'not received', 'nhi aaya': 'not received', 'nahi aya': 'not received',
    'nhi aya': 'not received', 'nahi aa raha': 'not received', 'nhi aa rha': 'not received',
    'kat gaya': 'deducted', 'kat gya': 'deducted', 'kat gaye': 'deducted', 'kat gye': 'deducted',
    'kaat liya': 'deducted', 'kat liya': 'deducted', 'cut gaya': 'deducted', 'cut gya': 'deducted',
    'chal nahi raha': 'not working', 'chal nhi rha': 'not working', 'nahi chal raha': 'not working',
    'nhi chal rha': 'not working', 'kaam nahi': 'not working', 'kaam nhi': 'not working',
    'band ho gaya': 'stopped', 'band ho gya': 'stopped', 'band hai': 'stopped',
    'raha hai': ' ', 'rha hai': ' ', 'rahi hai': ' ', 'rhi hai': ' ',
    'ho raha': ' ', 'ho rha': ' ', 'ho gaya': ' ', 'ho gya': ' ',
    'paisa': 'money', 'paise': 'money', 'paisaa': 'money', 'pese': 'money',
    'wapas': 'back', 'vapas': 'back', 'vaapas': 'back', 'wapis': 'back', 'vapis': 'back',
    'nahi': 'not', 'nahin': 'not', 'nhi': 'not', 'nai': 'not',
    'bekar': 'useless', 'bekaar': 'useless', 'bakwas': 'useless', 'bakwaas': 'useless',
    'ghatiya': 'useless', 'faltu': 'useless',
    'accha': 'good', 'acha': 'good', 'achha': 'good', 'achcha': 'good', 'badhiya': 'good', 'badiya': 'good',
    'bahut': 'very', 'bohot': 'very', 'bhut': 'very', 'bahot': 'very', 'bohut': 'very',
    'kyu': 'why', 'kyun': 'why', 'kyon': 'why', 'kab': 'when', 'kaise': 'how', 'kese': 'how',
    'khata': 'account',
}
BRANDS = {
    'phone pe': 'phonepe', 'phone pay': 'phonepe', 'phonepay': 'phonepe', 'phone-pe': 'phonepe',
    'google pay': 'googlepay', 'g pay': 'googlepay', 'gpay': 'googlepay', 'g-pay': 'googlepay', 'tez': 'googlepay',
    'pay tm': 'paytm',
}
NEGATIONS = {'not', 'no', 'never', 'nor', 'without', 'nothing'}
HINGLISH_STOP = {'hai', 'hain', 'ho', 'ka', 'ki', 'ke', 'se', 'mein', 'ko', 'bhi', 'ye', 'yeh', 'wo', 'woh',
                 'kar', 'karo', 'kiya', 'diya', 'liya', 'tha', 'thi', 'hua', 'hui', 'gaya', 'gya', 'raha',
                 'rha', 'rahi', 'aur', 'ya', 'par', 'ek', 'mera', 'meri', 'mere', 'apna', 'koi', 'kuch',
                 'sab', 'bhai', 'ji'}
BRAND_STOP = {'phonepe', 'googlepay', 'paytm', 'bhim', 'app', 'apps', 'application'}
STOP = (set(stopwords.words('english')) | HINGLISH_STOP | BRAND_STOP) - NEGATIONS

print('Loaded', len(df), 'reviews;', len(SLANG) + len(HINGLISH) + len(BRANDS), 'dictionary entries')

Loaded 86056 reviews; 125 dictionary entries


Cell P2: light cleaning, junk filter and cleaning funnel

In [ ]:
def build_replacer(mapping):
    keys = sorted(mapping, key=len, reverse=True)
    pat = re.compile(r'\b(' + '|'.join(map(re.escape, keys)) + r')\b')
    return lambda t: pat.sub(lambda m: mapping[m.group(1)], t)

replace_words = build_replacer({**BRANDS, **HINGLISH, **SLANG})

def light_clean(t):
    t = t.replace('\u2019', "'").replace('\u2018', "'")
    t = emoji.demojize(t, delimiters=(' ', ' ')).replace('_', ' ')
    t = t.lower()
    t = re.sub(r'[a-z-]*\s?skin\s?tone', ' ', t)
    t = re.sub(r'http\S+|www\.\S+|\S+@\S+|\b\d{10,12}\b', ' ', t)
    t = re.sub(r'([a-z])\1{2,}', r'\1\1', t)
    for pat, rep in CONTRACTIONS:
        t = re.sub(pat, rep, t)
    t = replace_words(t)
    return re.sub(r'\s+', ' ', t).strip()

INDIC = r'[\u0900-\u0DFF]'   # Hindi, Bengali, Punjabi, Gujarati, Odia, Tamil, Telugu, Kannada, Malayalam scripts

df['text_light'] = df['review_text'].apply(light_clean)
df['emojis'] = df['review_text'].apply(lambda t: ''.join(emoji.distinct_emoji_list(t)))
df['words_no_emoji'] = df['review_text'].apply(lambda t: len(emoji.replace_emoji(t, '').split()))
df['is_long'] = df['words_no_emoji'] >= 5
df['has_indic'] = df['review_text'].str.contains(INDIC, regex=True)
latin_words = df['text_light'].str.count(r'[a-z]{2,}')
df['indic_only'] = df['has_indic'] & (latin_words == 0)
df['is_junk'] = (latin_words == 0) & ~df['has_indic']

funnel = df.groupby('app').agg(scraped=('review_id', 'size'),
                               junk_removed=('is_junk', 'sum'),
                               indic_script_only=('indic_only', 'sum'))
funnel['kept'] = funnel['scraped'] - funnel['junk_removed']
usable = df[~df.is_junk & ~df.indic_only]
funnel['english_text_usable'] = usable.groupby('app').size()
funnel['long_usable'] = usable.groupby('app')['is_long'].sum()
funnel.loc['TOTAL'] = funnel.sum()
print(funnel)
print('\nJunk examples:', df[df.is_junk]['review_text'].head(15).tolist())
print('\nIndian-script examples:', df[df.indic_only]['review_text'].head(8).tolist())

df = df[~df.is_junk].copy()

            scraped  junk_removed  indic_script_only   kept  english_text_usable  long_usable
app                                                                                          
BHIM          10000             2                 82   9998                 9916         6951
Google Pay    18221            35                207  18186                17979         4930
Paytm         27614            13                272  27601                27329         4841
PhonePe       30221            27                268  30194                29926         4829
TOTAL         86056            77                829  85979                85150        21551

Junk examples: ['№##', '★★★★★', 'w /L', '1 n......', '₹1,500', 'p ,', '... ..', '🇳\u200c🇮\u200c🇨\u200c🇪\u200c', ', >>A', 'o.k', '(\u3000＾ω＾)(\u3000＾▽＾)( ´;ﾟ;∀;ﾟ;)(\u3000＾ω＾)( =＾ω＾)(ﾟーﾟ*)(*￣∇￣)ノ(*σ>∀<)σ(//∇//)', '𝓷𝓮𝓪𝓳𝓭𝓫𝓾', 'k,', '51//,-6%1u4', '𝓮𝓲𝓽𝓮𝓪𝓱 𝓳𝓪𝓽𝓪𝓿']

Indian-script examples: ['भरोसे मंद', 'उधार देदो कोई लौटा दूंगा ।', 'मोबाइल रिचार

Cell P3: full cleaning (lemmatising), save, and check the result

In [ ]:
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])
LEMMA_FIX = {'datum': 'data'}

def full_clean_all(texts):
    texts = [re.sub(r'[^a-z\s]', ' ', t) for t in texts]
    out = []
    for doc in nlp.pipe(texts, batch_size=2000):
        out.append(' '.join(LEMMA_FIX.get(w.lemma_, w.lemma_) for w in doc
                            if w.is_alpha and w.lower_ not in STOP
                            and (len(w) > 2 or w.lower_ in NEGATIONS)))
    return out

no_emoji = df['review_text'].apply(lambda t: light_clean(emoji.replace_emoji(t, ' ')))
df['text_full'] = full_clean_all(no_emoji.tolist())

df.drop(columns=['is_junk']).to_parquet(f'{BASE}/data/processed/clean_reviews.parquet', index=False)
print('Saved', len(df), 'reviews to clean_reviews.parquet')

eng = df[~df.indic_only]
print('English-text reviews with empty text_full:', (eng['text_full'].str.len() == 0).sum())

pd.set_option('display.max_colwidth', 120)
print(eng[eng.is_long & (eng.rating <= 2)].sample(6, random_state=7)[['review_text', 'text_light', 'text_full']])

from sklearn.feature_extraction.text import CountVectorizer
top = {}
for a in ['BHIM', 'Google Pay', 'Paytm', 'PhonePe']:
    t = eng[(eng.app == a) & (eng.rating <= 2) & eng.is_long]['text_full']
    cv = CountVectorizer(ngram_range=(2, 2), min_df=5)
    X = cv.fit_transform(t)
    s = pd.Series(X.sum(axis=0).A1, index=cv.get_feature_names_out()).nlargest(12)
    top[a] = [f'{k} ({v})' for k, v in s.items()]
print(pd.DataFrame(top))

Saved 85979 reviews to clean_reviews.parquet
English-text reviews with empty text_full: 4737
                                                                                                                   review_text  \
44919                                                         worse experience,charges extra for recharge.not at all optimized   
34746                                                                          falthu app unble to type otps app is very waste   
37298  auto reading of messages are not working but all other apps are taking auto reading otp but this does not ....and ev...   
67257  till now they were saying that I need to raise a ticket to report and now they r asking me to login to raise a ticke...   
80875                                                                       Aaj thak ek rupaayi kaa bii reward nahi milgaya...   
33687  Could you please explain what type of scam this is? I'm trying to delete my UPI Lite account, but the app keeps fail... 